In [30]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

In [31]:
torch.cuda.is_available()
df = pd.read_csv("C38_ready-forSure.csv")

features = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'days_offset']
targets = ['x_error', 'y_error', 'z_error']

X = df[features].values
y = df[targets].values

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
split_idx = int(len(df) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

df_test = df.iloc[split_idx:].reset_index(drop=True)

scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train)
y_test = scaler_y.transform(y_test)

# X_tensor = torch.tensor(X, dtype=torch.float32)
# y_tensor = torch.tensor(y, dtype=torch.float32)
# dataset = TensorDataset(X_tensor, y_tensor)

train_dataset = TensorDataset(  
    torch.tensor(X_train, dtype=torch.float32), 
    torch.tensor(y_train, dtype=torch.float32)
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32), 
    torch.tensor(y_test, dtype=torch.float32)
)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)


In [32]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(7, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128,128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        return self.network(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model().to(device=device)

loss_fn = nn.L1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-2)

epochs = 1000
for epoch in range(epochs):
    model.train()
    ov_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        y_pred = model(batch_X)
        loss = loss_fn(y_pred, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ov_loss += loss.item() * batch_X.size(0)

    epoch_loss = ov_loss / len(train_dataset)

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")

Epoch [100/1000], Loss: 0.0223
Epoch [200/1000], Loss: 0.0273
Epoch [300/1000], Loss: 0.0158
Epoch [400/1000], Loss: 0.0128
Epoch [500/1000], Loss: 0.0115
Epoch [600/1000], Loss: 0.0103
Epoch [700/1000], Loss: 0.0126
Epoch [800/1000], Loss: 0.0093
Epoch [900/1000], Loss: 0.0100
Epoch [1000/1000], Loss: 0.0108


In [33]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        y_pred = model(batch_X)
        all_preds.append(y_pred.cpu().numpy())
        all_targets.append(batch_y.cpu().numpy())

preds_array = np.vstack(all_preds)
targets_array = np.vstack(all_targets)

preds_km = scaler_y.inverse_transform(preds_array)
targets_km = scaler_y.inverse_transform(targets_array)

err3d_model = np.linalg.norm(preds_km - targets_km, axis=1)
err3d_raw_sgp4 = np.linalg.norm(targets_km, axis=1)

df_test['Err3D_Model'] = err3d_model
df_test['Err3D_SGP4'] = err3d_raw_sgp4

print("\n=== PODSUMOWANIE OGÓLNE 3D [km] ===")
print(f"SGP4  -> Średnia: {err3d_raw_sgp4.mean():.3f} | Mediana: {np.median(err3d_raw_sgp4):.3f} | 90th perc (ogon): {np.percentile(err3d_raw_sgp4, 90):.3f}")
print(f"Model -> Średnia: {err3d_model.mean():.3f} | Mediana: {np.median(err3d_model):.3f} | 90th perc (ogon): {np.percentile(err3d_model, 90):.3f}")

print("\n=== BŁĄD 3D W ROZBICIU NA DAYS_OFFSET [km] ===")
summary = df_test.groupby('days_offset').apply(lambda g: pd.Series({
    'Liczba_Próbek': len(g),
    'Mediana_SGP4': g['Err3D_SGP4'].median(),
    'Mediana_Model': g['Err3D_Model'].median(),
    'Średnia_SGP4': g['Err3D_SGP4'].mean(),
    'Średnia_Model': g['Err3D_Model'].mean(),
    '90perc_SGP4': np.percentile(g['Err3D_SGP4'], 90),
    '90perc_Model': np.percentile(g['Err3D_Model'], 90)
}))

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(summary.round(3).to_string())


=== PODSUMOWANIE OGÓLNE 3D [km] ===
SGP4  -> Średnia: 18.844 | Mediana: 2.556 | 90th perc (ogon): 11.150
Model -> Średnia: 19.007 | Mediana: 2.788 | 90th perc (ogon): 11.971

=== BŁĄD 3D W ROZBICIU NA DAYS_OFFSET [km] ===
             Liczba_Próbek  Mediana_SGP4  Mediana_Model  Średnia_SGP4  Średnia_Model  90perc_SGP4  90perc_Model
days_offset                                                                                                    
0                    106.0         1.253          1.803         1.398          2.077        2.439         3.279
1                    107.0         1.117          1.651         3.103          3.551        2.518         2.995
2                    107.0         1.091          1.608         4.262          4.773        2.157         3.013
3                    106.0         1.339          1.724         5.712          6.180        2.155         3.147
4                    106.0         2.005          1.904         9.330          9.701        3.031        

C:\Users\kozak\AppData\Local\Temp\ipykernel_20204\2958093815.py:29: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = df_test.groupby('days_offset').apply(lambda g: pd.Series({


In [34]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Szybki test na obecnych danych
X_train, X_test, y_train, y_test = train_test_split(X, y[:, 0], test_size=0.2, shuffle=False) # Chronologicznie!

gbr = GradientBoostingRegressor(n_estimators=100, random_state=42)
gbr.fit(X_train, y_train)
preds = gbr.predict(X_test)

print("MAE Gradient Boosting dla osi X:", mean_absolute_error(y_test, preds))

MAE Gradient Boosting dla osi X: 18.674813965664065


In [35]:
N = 10  # ile ostatnich wierszy pokazać

view = pd.DataFrame({
    'target_epoch':  df_test['target_epoch'].values,
    'days_offset':   df_test['days_offset'].values,

    # prawdziwy błąd SGP4 (prawda_SP3 - SGP4), bez korekty
    'SGP4_dx': targets_km[:, 0],
    'SGP4_dy': targets_km[:, 1],
    'SGP4_dz': targets_km[:, 2],

    # korekta przewidziana przez model
    'Model_dx': preds_km[:, 0],
    'Model_dy': preds_km[:, 1],
    'Model_dz': preds_km[:, 2],

    # odległość od prawdy w 3D: bez korekty vs z korektą
    'Dist_SGP4 [km]':  np.linalg.norm(targets_km, axis=1),
    'Dist_Model [km]': np.linalg.norm(targets_km - preds_km, axis=1),
})
view['Poprawa [km]'] = view['Dist_SGP4 [km]'] - view['Dist_Model [km]']  # >0 = model pomógł

pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', None)
print(view.tail(N).round(3).to_string(index=False))

       target_epoch  days_offset     SGP4_dx     SGP4_dy    SGP4_dz  Model_dx  Model_dy  Model_dz  Dist_SGP4 [km]  Dist_Model [km]  Poprawa [km]
2025-11-20 12:00:00            1 -136.947998  -37.188000 132.688004    -0.028     0.379    -0.389      194.278000       194.597000        -0.319
2025-11-21 12:00:00            2 -238.695007  -50.976002 216.042007    -0.126     0.323    -0.343      325.957001       326.141998        -0.186
2025-11-22 12:00:00            3 -338.399994  -64.097000 302.179993    -0.508     0.263     0.067      458.187988       457.804993         0.382
2025-11-23 12:00:00            4 -435.884003  -76.694000 390.954987    -1.467     0.064     0.965      590.526978       588.814026         1.713
2025-11-24 12:00:00            5 -530.942993  -88.875999 482.141998    -2.707    -0.199     2.050      722.676025       719.294983         3.381
2025-11-25 12:00:00            6 -623.390991 -100.700996 575.479004    -3.051    -0.163     2.237      854.361023       850.609009

In [36]:
view = pd.DataFrame({
    'tle_epoch':   df_test['tle_epoch'].values,
    'target_epoch': df_test['target_epoch'].values,
    'days_offset': df_test['days_offset'].values,

    # surowy wynik SGP4 (pozycja, km)
    'SGP4_x': df_test['x'].values,
    'SGP4_y': df_test['y'].values,
    'SGP4_z': df_test['z'].values,

    # surowy wynik modelu (przewidziany błąd, km)
    'Model_x': preds_km[:, 0],
    'Model_y': preds_km[:, 1],
    'Model_z': preds_km[:, 2],
})
print(view.tail(10).round(3).to_string(index=False))

                 tle_epoch        target_epoch  days_offset     SGP4_x    SGP4_y     SGP4_z  Model_x  Model_y  Model_z
2025-11-19 14:15:02.708640 2025-11-20 12:00:00            1 -23015.093 24775.589 -25217.043   -0.028    0.379   -0.389
2025-11-19 14:15:02.708640 2025-11-21 12:00:00            2 -23102.155 25103.478 -24812.804   -0.126    0.323   -0.343
2025-11-19 14:15:02.708640 2025-11-22 12:00:00            3 -23178.537 25436.190 -24402.264   -0.508    0.263    0.067
2025-11-19 14:15:02.708640 2025-11-23 12:00:00            4 -23244.403 25773.271 -23985.344   -1.467    0.064    0.965
2025-11-19 14:15:02.708640 2025-11-24 12:00:00            5 -23299.892 26114.200 -23562.052   -2.707   -0.199    2.050
2025-11-19 14:15:02.708640 2025-11-25 12:00:00            6 -23345.083 26458.409 -23132.519   -3.051   -0.163    2.237
2025-11-19 14:15:02.708640 2025-11-26 12:00:00            7 -23379.961 26805.306 -22697.022   -3.505   -0.068    2.392
2025-11-19 14:15:02.708640 2025-11-27 12:00:00  

In [37]:
for a in [0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]:
    e = np.linalg.norm(targets_km - a * preds_km, axis=1)
    print(a, e.mean().round(3), np.median(e).round(3), np.percentile(e, 90).round(3))

0 18.844 2.556 11.15
0.25 18.583 2.34 10.494
0.5 18.506 2.229 10.547
0.75 18.65 2.425 10.993
1.0 19.007 2.788 11.971
1.5 20.075 3.649 14.561
2.0 21.403 4.584 18.046
